In [9]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

golub = fetch_openml(data_id=31, as_frame=False)
X = golub.data
y = golub.target

# Encode target labels to 0/1
le = LabelEncoder()
y = le.fit_transform(y)

# Convert categorical features to numeric using one-hot encoding
categorical_mask = np.array([isinstance(X[0, i], str) for i in range(X.shape[1])])
X_categorical = X[:, categorical_mask]
X_numeric = X[:, ~categorical_mask].astype(float)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_categorical_encoded = ohe.fit_transform(X_categorical)

X = np.hstack([X_numeric, X_categorical_encoded])

In [10]:
from sklearn.feature_selection import f_classif
from typing import cast
from numpy.typing import ArrayLike

f_scores, _ = f_classif(cast(ArrayLike, X), y)
f_ranking = np.argsort(f_scores)[::-1]
print("Feature ranking by F-score (highest to lowest):", f_ranking)

Feature ranking by F-score (highest to lowest): [10  8  0 12 28  1 15 51 11 30 45  7 46 48 22 33 25 20 47 52  4 29 60 59
 50 39 32 36  2 27 18 40 34 41 49 37  5  9 14 53 35 24 58 57 16 21 26 56
 19 23 38 54 13 43 31 17 44 55  6  3 42]


In [11]:
from sklearn.metrics import roc_auc_score

roc_aucs = []
for i in range(X.shape[1]):
    roc_aucs.append(-roc_auc_score(y, X[:, i]))

roc_auc_ranking = np.argsort(roc_aucs)[::-1]
print("Feature ranking by opposite of ROC-AUC (highest to lowest):", roc_auc_ranking)

Feature ranking by opposite of ROC-AUC (highest to lowest): [ 0  8 28  7  1 45 20 33  2 52 36 47 11 15 50 14 57 60 18 53 40 37 49 16
 35 19 26 43 31 13 21 44 23  3 17 55 42  6 24 38 54 56  9 41 59 27 58 29
  5 34 32 25 39 48 22 30 46 51  4 12 10]


In [12]:
from scipy.stats import pearsonr

pearson_scores = [abs(cast(float, pearsonr(X[:, i], y)[0])) for i in range(X.shape[1])]
pearson_ranking = np.argsort(pearson_scores)[::-1]
print("Feature ranking by absolute Pearson correlation (highest to lowest):", pearson_ranking)

Feature ranking by absolute Pearson correlation (highest to lowest): [10  8  0 12 28  1 15 51 11 30 45  7 46 48 22 33 25 20 47 52  4 29 59 60
 50 39 32 36  2 27 18 40 34 41 49 37  5  9 14 53 35 24 58 57 16 21 26 56
 19 23 38 54 13 43 31 17 44 55  6  3 42]


In [13]:
top_n = 3

top_f_features = f_ranking[:top_n]
top_roc_auc_features = roc_auc_ranking[:top_n]
top_pearson_features = pearson_ranking[:top_n]

selected_features = np.unique(np.concatenate([top_f_features, top_roc_auc_features, top_pearson_features]))
print("Union of selected features:", selected_features)

Union of selected features: [ 0  8 10 28]


In [14]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import pandas as pd

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

results = []

# All features
start = time.time()
clf_all = SVC()
clf_all.fit(X_train, y_train)
y_pred_all = clf_all.predict(X_test)
acc_all = accuracy_score(y_test, y_pred_all)
elapsed_all = time.time() - start

results.append({
    "Features": "All",
    "Accuracy": round(acc_all, 3),
    "Time (s)": float(f"{elapsed_all:.2g}")
})

# Selected features
start = time.time()
clf_sel = SVC()
clf_sel.fit(X_train[:, selected_features], y_train)
y_pred_sel = clf_sel.predict(X_test[:, selected_features])
acc_sel = accuracy_score(y_test, y_pred_sel)
elapsed_sel = time.time() - start

results.append({
    "Features": "Selected",
    "Accuracy": round(acc_sel, 3),
    "Time (s)": float(f"{elapsed_sel:.2g}")
})

# Display results
df_results = pd.DataFrame(results)
print(df_results)

   Features  Accuracy  Time (s)
0       All     0.712     0.050
1  Selected     0.708     0.015
